# 💻 NanoGPT Lab — Codex & Claude Caliber Coding AI Training
### Fine-Tune LLaMA-3.2 (1B/3B) into an Elite Coding Model on Free Google Colab GPU

This notebook trains your AI model to write production-grade code, solve complex algorithms, debug errors, and explain system architecture just like **Claude 3.5 Sonnet** and **OpenAI Codex**.

**Hardware**: Free Google Colab **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).
**Output**: Exports a 4-bit quantized GGUF (`Llama-3.2-1B-Coder.Q4_K_M.gguf`) to run locally on your laptop CPU at ~20 tokens/sec.

In [ ]:
# Step 1: Verify Google Colab GPU
!nvidia-smi

In [ ]:
# Step 2: Install High-Performance Fine-Tuning Stack (Unsloth + PyTorch)
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# Step 3: Load LLaMA-3.2 Base Architecture in 4-bit Quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model_name = "unsloth/Llama-3.2-1B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("Base model loaded successfully into GPU VRAM!")

In [ ]:
# Step 4: Attach LoRA Fine-Tuning Adapters across Attention & MLP Layers
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("LoRA Adapters initialized!")

In [ ]:
# Step 5: Load Master Coding & Reasoning Dataset (ChatGPT / Claude / Codex Caliber)
import os
import json
import urllib.request
from datasets import load_dataset, Dataset

# 1. Download or load the curated chatgpt_dataset.json (100+ verified coding & CS seeds)
dataset_file = "chatgpt_dataset.json"
if not os.path.exists(dataset_file):
    repo_url = "https://raw.githubusercontent.com/Prem759-0/NanoGPT/main/chatgpt_dataset.json"
    try:
        print(f"Fetching {dataset_file} from GitHub repository...")
        urllib.request.urlretrieve(repo_url, dataset_file)
    except Exception as e:
        print(f"Could not auto-download from GitHub: {e}")

training_examples = []
if os.path.exists(dataset_file):
    with open(dataset_file, "r", encoding="utf-8") as f:
        records = json.load(f)
        print(f"Successfully loaded {len(records)} curated master examples from {dataset_file}!")
        for r in records:
            inst = r.get("instruction", "").strip()
            resp = r.get("response", "").strip()
            if inst and resp:
                fmt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou are an expert coding and reasoning AI assistant of Claude and ChatGPT caliber.<|eot_id|><|start_header_id|>user<|end_header_id|>\n{inst}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n{resp}<|eot_id|>"
                training_examples.append({"text": fmt})

# 2. Supplement with CodeAlpaca for high-diversity algorithmic coverage
try:
    alpaca = load_dataset("sahil2801/CodeAlpaca-20k", split="train[:1000]")
    for sample in alpaca:
        inst = sample["instruction"]
        inp = sample.get("input", "")
        out = sample["output"]
        full_inst = f"{inst}\nInput: {inp}" if inp else inst
        fmt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou are an expert coding assistant of Codex & Claude caliber.<|eot_id|><|start_header_id|>user<|end_header_id|>\n{full_inst}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n{out}<|eot_id|>"
        training_examples.append({"text": fmt})
    print(f"Added 1,000 CodeAlpaca samples. Total training corpus: {len(training_examples)} instructions!")
except Exception as e:
    print(f"Using curated corpus only: {e}")

dataset = Dataset.from_list(training_examples)
print(f"Training dataset ready with {len(dataset)} examples!")


In [ ]:
# Step 6: Train the Model with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        output_dir = "outputs_coder",
    ),
)

trainer.train()
print("Fine-tuning complete! Your Coding AI is ready!")

In [ ]:
# Step 7: Export to GGUF format for Laptop CPU execution
model.save_pretrained_gguf("Llama-3.2-1B-Coder-Q4_K_M", tokenizer, quantization_method = "q4_k_m")
print("GGUF model exported successfully!")

# Optional: Download directly to your browser or save to Google Drive
try:
    from google.colab import files
    # Look for the generated .gguf file
    for f in os.listdir('.'):
        if f.endswith('.gguf') or 'Llama-3.2-1B-Coder' in f:
            print(f"Model artifact ready: {f}")
            # files.download(f)  # Uncomment to trigger direct browser download
except Exception:
    pass
